## Instalación de librerías

In [4]:
!pip install -q scrapy
#!pip install -q newspaper4k
!pip install -q newspaper3k
!pip install -q lxml_html_clean


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\rebec\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\rebec\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\rebec\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Articulos de mapas de sitio usando scrapy

In [5]:
!scrapy startproject news_scraper

Error: scrapy.cfg already exists in C:\Users\rebec\OneDrive - Universidad de Oviedo\Escritorio\TFG\news_scraper


## Extracción de URLs

In [6]:
%cd news_scraper

c:\Users\rebec\OneDrive - Universidad de Oviedo\Escritorio\TFG\news_scraper


In [ ]:
%%writefile "news_extractor_spider.py"
import scrapy
from newspaper import build, Config, Article
import requests
from urllib.parse import urljoin, urlparse
import os
import time
import re
from datetime import datetime
import gzip
from scrapy.selector import Selector
import dateutil

class NewsUrlExtractorSpider(scrapy.Spider):
    name = 'news_extractor'

    def __init__(self, date="2025-01-01", *args, **kwargs):
        super(NewsUrlExtractorSpider, self).__init__(*args, **kwargs)
        self.from_date = datetime.strptime(date, '%Y-%m-%d').date()
        self.invalid_url_words = {'section', 'tag', 'template',
                                  'category', 'author', 'page-sitemap'
                                  'categories', 'video', 'image', 'temas',
								                  'live', 'microsite', 'focus', 'blog',
                                  'ocio', 'cine', 'board', 'character'}


    # Lista de páginas web
    start_urls=[
        'https://www.elcomercio.es/',
        'https://www.lne.es/',
        'https://www.lavanguardia.com/',
        'https://www.larazon.es/',
        'https://www.rtpa.es/',
        'https://www.europapress.es/',
        'https://www.abc.es/',
        'https://www.20minutos.es/',
        'https://www.elperiodico.com/',
        'https://www.eldiario.es/',
        'https://www.elconfidencial.com/',
        'https://www.culturalgijonesa.org/',
        'https://www.elespanol.com/',
        'https://www.nortes.me/',
        'https://www.asturiasmundial.com/',
        'https://www.tribunasalamanca.com/',
        'https://migijon.com/',
        'https://www.telecinco.es/',
        'https://www.laprovincia.es/',
        'https://www.laopiniondemalaga.es/',
        'https://www.elfielato.es/',
        'https://www.teleprensa.com/',
        'https://www.infobae.com/',
        'https://www.lavozdeasturias.es/', #newspaper
        'https://cualia.es/', #newspaper
        'https://www.lavozdegalicia.es/', # newspaper
        'http://www.gentedigital.es/', #newspaper
    ]

    # Metadata: nombre y sitemap
    metadata_urls={
        'https://www.elcomercio.es/': {'nombre': 'El Comercio'},
        'https://www.lne.es/': {'nombre': 'La Nueva España'},
        'https://www.lavanguardia.com/': {'nombre': 'La Vanguardia',
                                          'sitemap': 'sitemap-google-news.xml'},
        'https://www.larazon.es/': {'nombre': 'La Razón'},
        'https://www.rtpa.es/': {'nombre': 'Radiotelevisión del Principado de Asturias (RTPA)',
                                 'sitemap': 'sitemap-noticias.xml'},
        'https://www.europapress.es/': {'nombre': 'Europa Press'},
        'https://www.abc.es/': {'nombre': 'ABC'},
        'https://www.20minutos.es/': {'nombre': '20 Minutos',
                                      'sitemap': 'sitemap-google-news.xml'},
        'https://www.elperiodico.com/' : {'nombre': 'El Periódico',
                                          'sitemap': 'google-news.xml'},
        'https://www.eldiario.es/' : {'nombre': 'ElDiario.es'},
        'https://www.lavozdeasturias.es/': {'nombre': 'La Voz de Asturias'}, #newspaper
        'https://www.elconfidencial.com/': {'nombre': 'El Confidencial',
                                            'sitemap': 'newsitemap_4.xml'},
        'https://cualia.es/': {'nombre': 'Cualia'}, #newspaper
        'https://www.culturalgijonesa.org/': {'nombre': 'Cultural Gijonesa'},
        'https://www.elespanol.com/' : {'nombre': 'El Español',
                                        'sitemap': 'sitemap_google_news.xml'},
        'https://www.nortes.me/': {'nombre': 'Nortes'},
        'https://www.lavozdegalicia.es/': {'nombre': 'La Voz de Galicia'},
        'https://www.asturiasmundial.com/': {'nombre': 'Asturias Mundial'},
        'https://www.tribunasalamanca.com/': {'nombre': 'Tribuna Salamanca'},
        'https://migijon.com/': {'nombre': 'Mi Gijón'},
        'http://www.gentedigital.es/' : {'nombre': 'Gente Digital'}, #newspaper
        'https://www.infobae.com/' : {'nombre': 'Infobae',
                                      'sitemap': 'arc/outboundfeeds/news-sitemap2/'},
        'https://www.telecinco.es/' : {'nombre': 'Telecinco'},
        'https://www.laprovincia.es/' : {'nombre': 'La Provincia'},
        'https://www.laopiniondemalaga.es/': {'nombre': 'La Opinión de Málaga'},
        'https://www.elfielato.es/': {'nombre': 'El Fielato y El Nora'},
        'https://www.teleprensa.com/': {'nombre': 'Teleprensa'}
    }


    custom_settings = {
        'USER_AGENT': "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:98.0) Gecko/20100101 Firefox/98.0"
    }


    # Para extracción de xmls
    namespaces = {
        'ns': 'http://www.sitemaps.org/schemas/sitemap/0.9',  # Default namespace
        'news': 'http://www.google.com/schemas/sitemap-news/0.9'
    }

    def get_base_url(self, url):
        parsed_url = urlparse(url)
        return f"{parsed_url.scheme}://{parsed_url.netloc}/"

    def normalize_date(self, date_string):
        try:
            # Conviertir la cadena de fechas en un objeto datetime
            parsed_date = dateutil.parser.parse(date_string)
            # Formatea el objeto datetime como 'YYYY-MM-DD'
            #return parsed_date.strftime('%Y-%m-%d')
            return parsed_date.date()
        except (ValueError, TypeError):
            raise ValueError(f"Formato de fecha inválido: '{date_string}'")

    def is_valid_xml_url(self, sitemap_url):
        for invalid_word in self.invalid_url_words:
            if invalid_word in sitemap_url:
                return False

        pattern = r'\b(1[0-9]{3}|20[0-1][0-9]|202[0-4])\b' # Matches 1000-1999, 2000-2019, and 2020-2024
        match_url = re.search(pattern, sitemap_url)
        if match_url:
            return False
        else:
            return True

    def parse(self, response):#se construye la url del archivo robots.txt para buscar los sidemaps
        robots_url = urljoin(response.url, "/robots.txt")
        print(f'Robots URL: {robots_url}')
        yield scrapy.Request(robots_url, callback=self.parse_robots, meta={'domain': response.url})


    def parse_robots(self, response):
        if response.status != 200: #verificamos que robots sea accesible
            print(f"Error al acceder a la url: {response.status}")
            return

        domain = response.meta['domain']
        domain_base = self.get_base_url(domain)
        current_sitemap_urls = set() #[]

        # Extraer urls de sitemaps de robots.txt
        for line in response.text.splitlines():
            if line.lower().startswith('sitemap:'):
                sitemap_url = line.split(':', 1)[1].strip()
                parsed_url = urlparse(sitemap_url)
                file_path = parsed_url.path
                file_name, file_extension = os.path.splitext(file_path)
                if file_extension.lower() == ".xml" and self.is_valid_xml_url(sitemap_url):
                    current_sitemap_urls.add(sitemap_url)

        if len(current_sitemap_urls) > 0:#Primero, si en robots apaerecen sidemaps
            print(f"Se encontraron los siguientes enlaces xml: {current_sitemap_urls}")
            for sitemap_url in current_sitemap_urls:
                print(f"Accediendo al siguiente enlace... {sitemap_url}")
                #yield scrapy.Request(sitemap_url, callback=self.parse_sitemap)  # Llamada recursiva
                yield scrapy.Request(sitemap_url, callback=lambda response: self.parse_sitemap(response, domain) )

        elif domain_base in self.metadata_urls and 'sitemap' in self.metadata_urls[domain_base]:#Segundo, sino, se accede al sidemap especificado
            print("No se encontraron enlaces xml en robots.txt.", end=" ")
            sitemap_name = self.metadata_urls[domain_base]['sitemap']
            #sitemap_name = self.sitemap_urls[domain]
            sitemap_url = urljoin(domain_base, sitemap_name)
            print(f"Accediendo al siguiente enlace especificado... {sitemap_url}")
            #yield scrapy.Request(sitemap_url, callback=self.parse_sitemap)
            yield scrapy.Request(sitemap_url, callback=lambda response: self.parse_sitemap(response, domain_base) )
        else:#Tercero, accedemos con newspaper
            print("No se encontraron enlaces xml. Accediendo a enlaces con librería Newspaper...")
            # Si no se encuentra ningún mapa del sitio,
            # utilizar Newspaper4k para obtener las URL de los artículos de noticias
            yield from self.get_news_urls(domain)



    def parse_sitemap(self, response, domain=None):#Accedemos a los sidemaps

        # Sitemaps anidados. Tag <sitemap>
        for sitemap in response.xpath('//ns:sitemap', namespaces=self.namespaces):
            sitemap_loc = sitemap.xpath('./ns:loc/text()', namespaces=self.namespaces).get()

            # Omitir sitemaps antiguos
            last_mod = sitemap.xpath('./ns:lastmod/text()', namespaces=self.namespaces).get()
            if last_mod:
                lastmod = self.normalize_date(last_mod)
                if lastmod < self.from_date:
                    continue

            # Verificar que la url es un archivo xml
            parsed_url = urlparse(sitemap_loc)
            file_path = parsed_url.path
            file_name, file_extension = os.path.splitext(file_path)

            if file_extension.lower() == ".xml" and self.is_valid_xml_url(sitemap_loc):#'section' not in sitemap_loc:
                #print(f'Se encontró otro xml dentro del archivo actual: {sitemap_loc}') ## lots of outputs
                if sitemap_loc:
                    #yield scrapy.Request(sitemap_loc, callback=self.parse_sitemap)  # Llamada recursiva
                    yield scrapy.Request(sitemap_loc, callback=lambda response: self.parse_sitemap(response, domain) )
            elif file_extension.lower() == '.gz'  and self.is_valid_xml_url(sitemap_loc):#'section' not in sitemap_loc:
                print(f"Se encontró archivo comprimido {sitemap_loc}")
                #yield scrapy.Request(sitemap_loc, callback=self.parse_sitemap_gz)
                yield scrapy.Request(sitemap_loc, callback=lambda response: self.parse_sitemap_gz(response, domain) )


        # Extraer datos de cada tag <url>
        for url in response.xpath('//ns:url', namespaces=self.namespaces):

            loc = url.xpath('./ns:loc/text()', namespaces=self.namespaces).get()
            title = url.xpath('./news:news/news:title/text()', namespaces=self.namespaces).get()
            publication_date = url.xpath('./news:news/news:publication_date/text()', namespaces=self.namespaces).get()
            fuente = url.xpath('./news:news/news:publication/news:name/text()', namespaces=self.namespaces).get()

            if loc is None:
                break

            title = "" if title is None else title
            if publication_date is None:
               publication_date = url.xpath('./ns:lastmod/text()', namespaces=self.namespaces).get()

            if publication_date is not None:
                publication_date_comp = self.normalize_date(publication_date)
                #publication_date_ = dateutil.parser.parse(publication_date).date()
                if publication_date_comp < self.from_date:
                    continue

            if domain is not None:
                domain_base = self.get_base_url(domain)
                if domain_base in self.metadata_urls:
                    fuente = self.metadata_urls[domain_base]['nombre']

            yield {
                'fuente': fuente,
                'url': loc,
                'titulo': title,
                'fecha_publicacion': publication_date,
            }

    def parse_sitemap_gz(self, response, domain=None):

        compressed_file = response.body
        decompressed_file = gzip.decompress(compressed_file).decode("utf-8")
        selector = Selector(text=decompressed_file, type="xml")
        #yield from self.parse_sitemap(selector)
        yield from self.parse_sitemap(selector, domain)

    def get_news_urls(self, domain):

        config = Config()
        config.request_timeout = 5
        config.language= 'es'
        config.thread_timeout_seconds = 5
        config.memoize_articles = False
        config.fetch_images = False
        config.follow_meta_refresh = True
        config.number_threads = 4
        config.browser_user_agent = self.custom_settings['USER_AGENT']
        config.headers = {
            "User-Agent": self.custom_settings['USER_AGENT'],
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.5",
            "Accept-Encoding": "gzip, deflate",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1",
            "Sec-Fetch-Dest": "document",
            "Sec-Fetch-Mode": "navigate",
            "Sec-Fetch-Site": "none",
            "Sec-Fetch-User": "?1",
            "Cache-Control": "max-age=0",
        }

        try:
            print(f'Extrayendo noticias con librería Newspaper para url: {domain}')
            start_time = time.time()
            paper = build(domain, config=config)
            print(f"--- {(time.time() - start_time):.2f}s segundos ---")
            articulos_urls = set(paper.article_urls())
            print(f'Se encontraron {len(articulos_urls)} enlaces de noticias.')

            # Obtener fuente
            if domain is not None:
                domain_base = self.get_base_url(domain)
                if domain_base in self.metadata_urls:
                    fuente = self.metadata_urls[domain_base]['nombre']
                else:
                    found_match = re.search(r"www\.(.*?)\.(.+)", domain)
                    fuente = ''
                    if found_match:
                        fuente = found_match.group(1)


            titulo, fecha_publicacion = '', None
            for articulo_url in articulos_urls:
                yield {
                    'fuente': fuente,
                    'url': articulo_url,
                    'titulo': titulo,
                    'fecha_publicacion': fecha_publicacion,
                }
        except Exception as e:
            print(f"No se pudieron obtener noticias de {domain}")

Writing news_extractor_spider.py


Copiar archivo spider a carpeta para ejecutar

In [15]:
# !cp news_extractor_spider.py news_scraper/spiders
!copy news_extractor_spider.py news_scraper\spiders\

        1 archivo(s) copiado(s).


Eliminar csv para generar uno nuevo

In [16]:
# Eliminar csv
import os
output_file_csv = "news_output.csv"
if os.path.exists(output_file_csv):
    os.remove(output_file_csv)

Ejecutar spyder de scrapy para gener el archivo csv con los enlaces

In [17]:
!scrapy crawl news_extractor -o news_output.csv -s LOG_ENABLED=False -s ROBOTSTXT_OBEY=False -a date=2025-02-01
#!scrapy crawl news_extractor -o news_output.csv

Robots URL: https://www.europapress.es/robots.txt
Robots URL: https://www.abc.es/robots.txt
Robots URL: https://www.eldiario.es/robots.txt
Robots URL: https://www.elconfidencial.com/robots.txt
Robots URL: https://www.elcomercio.es/robots.txt
Robots URL: https://www.20minutos.es/robots.txt
Robots URL: https://www.lne.es/robots.txt
Robots URL: https://www.elespanol.com/robots.txt
Robots URL: https://www.lavanguardia.com/robots.txt
Se encontraron los siguientes enlaces xml: {'https://www.europapress.es/sitemap.xml'}
Accediendo al siguiente enlace... https://www.europapress.es/sitemap.xml
No se encontraron enlaces xml. Accediendo a enlaces con librería Newspaper...
Extrayendo noticias con librería Newspaper para url: https://www.abc.es/
--- 72.48s segundos ---
Se encontraron 1548 enlaces de noticias.
No se encontraron enlaces xml en robots.txt. Accediendo al siguiente enlace especificado... https://www.elespanol.com/sitemap_google_news.xml
No se encontraron enlaces xml en robots.txt. Acced

## Filtrar articulos por fecha

In [31]:
import pandas as pd
from datetime import datetime, timedelta

# Cargar la data extraída
data_df = pd.read_csv("news_output.csv")

# Obtener la fecha de ayer y hoy
hoy = datetime.utcnow().strftime('%Y-%m-%d')
ayer = (datetime.utcnow() - timedelta(days=1)).date()

# Convertir las fechas de publicación
data_df['fecha_publicacion'] = pd.to_datetime(data_df['fecha_publicacion'], format='mixed', errors='coerce', utc=True)

# Convertir a solo fecha (sin hora)
data_df['fecha_publicacion'] = data_df['fecha_publicacion'].dt.date

# Filtrar solo las noticias de ayer y hoy
data_df = data_df[data_df['fecha_publicacion'].isin([ayer, datetime.utcnow().date()])]

# Eliminar duplicados y ordenar por fecha de publicación
data_df = data_df.sort_values(by='fecha_publicacion', ascending=False).drop_duplicates().reset_index(drop=True)


# Guardar el CSV con la fecha de hoy en el nombre
nombre_archivo = f"{hoy}.csv"
data_df.to_csv(nombre_archivo, index=False)

# Mostrar el nombre del archivo guardado
print(f"Archivo guardado como: {nombre_archivo}")

# Mostrar el resultado
data_df


Archivo guardado como: 2025-02-21.csv


,fuente,url,titulo,fecha_publicacion
0,El Comercio,https://www.elcomercio.es/culturas/fetehn-2025...,NaN,2025-02-21
1,ElDiario.es,https://www.eldiario.es/politica/ultima-hora-a...,NaN,2025-02-21
2,ElDiario.es,https://www.eldiario.es/vertele/noticias/bronc...,NaN,2025-02-21
3,ElDiario.es,https://www.eldiario.es/euskadi/blogs/viento-d...,NaN,2025-02-21
4,ElDiario.es,https://www.eldiario.es/castilla-la-mancha/pro...,NaN,2025-02-21
...,...,...,...,...
12195,La Razón,https://www.larazon.es/sociedad/mayor-peligro-...,"El mayor peligro, que necesite intubación en l...",2025-02-20
12196,La Razón,https://www.larazon.es/andalucia/psoe-montero-...,El PSOE de Montero retoma la idea de una «banc...,2025-02-20
12197,La Razón,https://www.larazon.es/andalucia/via-andaluza-...,"<![CDATA[La ""vía andaluza"" llega a Europa]]>",2025-02-20
12198,La Razón,https://www.larazon.es/lifestyle/moda/tributo-...,Del tributo a la DANA de María Lafuente al apr...,2025-02-20


In [19]:
data_df['fuente'].value_counts()

fuente
La Razón                                             1677
Teleprensa                                           1454
ElDiario.es                                          1040
El Periódico                                         1010
La Nueva España                                      1007
La Vanguardia                                         844
El Español                                            843
La Opinión de Málaga                                  801
El Confidencial                                       722
La Provincia                                          710
20 Minutos                                            653
El Comercio                                           608
Telecinco                                             409
Tribuna Salamanca                                     111
Radiotelevisión del Principado de Asturias (RTPA)     109
Infobae                                               100
El Fielato y El Nora                                   40
Mi Gijó

### Eliminar enlaces duplicados

In [20]:
def remover_url_duplicados(df):
    df['non_null_count'] = df.notnull().sum(axis=1)
    df = df.sort_values(by=['url', 'non_null_count'], ascending=[True, False])
    df = df.drop_duplicates(subset=['url'], keep='first')
    df = df.drop(columns=['non_null_count'])
    df = df.sort_values(by='fecha_publicacion', ascending=False).reset_index(drop=True)
    return df

data_df = remover_url_duplicados(data_df)
data_df

,fuente,url,titulo,fecha_publicacion
0,Mi Gijón,https://migijon.com/about-us/,NaN,2025-02-21
1,La Razón,https://www.larazon.es/andalucia/sevilla/amena...,<![CDATA[Amenaza de muerte con una navaja a un...,2025-02-21
2,La Razón,https://www.larazon.es/asturias/fernando-laser...,"Fernando Laserna Cocina, nombrado nuevo Fiscal...",2025-02-21
3,La Razón,https://www.larazon.es/asturias/ampliada-astur...,Ampliada en Asturias la temporada de caza a ma...,2025-02-21
4,La Razón,https://www.larazon.es/animales-mascotas/cucar...,<![CDATA[Las cucarachas 'patas arriba' no apar...,2025-02-21
...,...,...,...,...
9055,ElDiario.es,https://www.eldiario.es/canariasahora/turismo/...,NaN,2025-02-20
9056,La Razón,https://www.larazon.es/deportes/futbol/cuando-...,<![CDATA[¿A qué hora es el sorteo de octavos d...,2025-02-20
9057,La Razón,https://www.larazon.es/deportes/futbol/claves-...,"Las claves del nuevo Real Madrid: una bronca, ...",2025-02-20
9058,La Razón,https://www.larazon.es/deportes/futbol/champio...,<![CDATA[Real Madrid - City: Las redes y los '...,2025-02-20


In [21]:
data_df['fuente'].value_counts()

fuente
Teleprensa                                           1154
La Razón                                              870
La Vanguardia                                         844
El Español                                            843
El Confidencial                                       722
20 Minutos                                            653
El Periódico                                          624
ElDiario.es                                           592
La Provincia                                          563
La Opinión de Málaga                                  540
La Nueva España                                       516
Telecinco                                             409
El Comercio                                           327
Tribuna Salamanca                                     111
Radiotelevisión del Principado de Asturias (RTPA)     109
Infobae                                               100
Mi Gijón                                               36
Nortes 

## Acceder al contenido de las noticias

### Extracción con request

In [22]:
import requests
from bs4 import BeautifulSoup


# Lista de user-agents para probar
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",
    "Mozilla/5.0 AppleWebKit/537.36 (KHTML, like Gecko; compatible; Googlebot/2.1; +http://www.google.com/bot.html) Chrome/W.X.Y.Z Safari/537.36"
]

def extraer_texto_articulos_request(url):
    try:
        # Primer intento sin user-agent
        response = requests.get(url)

        # Si la respuesta no es exitosa o el contenido está vacío, intente con agentes de usuario
        if response.status_code != 200 or not response.content:
            for user_agent in user_agents:
                headers = {
                    "User-Agent": user_agent,
                    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
                    "Accept-Language": "en-US,en;q=0.5",
                    "Accept-Encoding": "gzip, deflate",
                    "Connection": "keep-alive",
                }
                response = requests.get(url, headers=headers)

                if response.status_code == 200 and response.content:
                    #print("Encontrado con user-agent")
                    break

        html_str = response.content
        soup = BeautifulSoup(html_str, 'lxml')

        # Remover footer
        footer = soup.find('footer')
        if footer:
            footer.decompose()

        text_content = soup.find_all('p')

        text = ''
        for i in text_content:
            text += i.text.strip() + " "

        return text.strip() if text.strip() else None

    except Exception as e:
        print(f"Error in url: {url}, error: {e}")
        return None

In [23]:
from concurrent.futures import ThreadPoolExecutor
import time

num_threads = 16

start_time = time.time()
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    data_df['texto'] = list(executor.map(extraer_texto_articulos_request, data_df['url']))
print(f"--- {(time.time() - start_time):.2f}s seconds ---")

Error in url: https://www.elperiodico.com/es/https:/www.elperiodico.com/politica/armas-espana-guerra-ucrania-sh/index.html, error: HTTPSConnectionPool(host='www.elperiodico.com', port=443): Max retries exceeded with url: /es/https:/www.elperiodico.com/politica/armas-espana-guerra-ucrania-sh/index.html (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001E6D217F2D0>, 'Connection to www.elperiodico.com timed out. (connect timeout=None)'))
Error in url: https://www.laprovincia.es/deportes/2025/02/21/roma-athletic-real-sociedad-manchester-octavos-europa-league-114547563.html, error: HTTPSConnectionPool(host='www.laprovincia.es', port=443): Max retries exceeded with url: /deportes/2025/02/21/roma-athletic-real-sociedad-manchester-octavos-europa-league-114547563.html (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001E6D29F8810>, 'Connection to www.laprovincia.es timed out. (connect timeout=None)'))
Error in url: https://www.l

In [24]:
data_df

,fuente,url,titulo,fecha_publicacion,texto
0,Mi Gijón,https://migijon.com/about-us/,NaN,2025-02-21,"Pablo Cardona, semifinalista en el P1 de Riad ..."
1,La Razón,https://www.larazon.es/andalucia/sevilla/amena...,<![CDATA[Amenaza de muerte con una navaja a un...,2025-02-21,Andalucía Sevilla España Local Opinión Intern...
2,La Razón,https://www.larazon.es/asturias/fernando-laser...,"Fernando Laserna Cocina, nombrado nuevo Fiscal...",2025-02-21,Asturias España Local Opinión Internacional Ec...
3,La Razón,https://www.larazon.es/asturias/ampliada-astur...,Ampliada en Asturias la temporada de caza a ma...,2025-02-21,Asturias España Local Opinión Internacional Ec...
4,La Razón,https://www.larazon.es/animales-mascotas/cucar...,<![CDATA[Las cucarachas 'patas arriba' no apar...,2025-02-21,Animales y mascotas España Local Opinión Inter...
...,...,...,...,...,...
9055,ElDiario.es,https://www.eldiario.es/canariasahora/turismo/...,NaN,2025-02-20,"Hoy hablamos de... Por favor, actualiza tus da..."
9056,La Razón,https://www.larazon.es/deportes/futbol/cuando-...,<![CDATA[¿A qué hora es el sorteo de octavos d...,2025-02-20,Deportes Fútbol España Local Opinión Internac...
9057,La Razón,https://www.larazon.es/deportes/futbol/claves-...,"Las claves del nuevo Real Madrid: una bronca, ...",2025-02-20,Deportes Fútbol España Local Opinión Internac...
9058,La Razón,https://www.larazon.es/deportes/futbol/champio...,<![CDATA[Real Madrid - City: Las redes y los '...,2025-02-20,Deportes Fútbol Champions League España Loca...


In [25]:
data_df['texto'].isna().sum()

24

In [26]:
(data_df['texto'] == '').sum()

0

In [27]:
data_df['texto'].iloc[-2]

"Deportes  Fútbol  Champions League España Local Opinión Internacional Economía Cultura Sociedad Deportes Gente A Tu Salud Televisión Ciencia Motor Lifestyle Emergente Premios LR Juegos y pasatiempos 25 Aniversario De Compras La tienda de La Razón Lea el periódico en PDF Stories La Razón Suplementos en PDF Promociones Directo Última hora del estado de salud del Papa Directo Última hora de la guerra de Ucrania: Musk arremete contra Zelenski y Trump le pide que firme ya el acuerdo de las tierras raras Champions League Todavía colea la resaca del partido de anoche en el Santiago Bernabéu, con la contundente victoria del Real Madrid sobre el Manchester City (3-1). Las portadas de toda la prensa europea hablan del partido, del gran papel de Mbappé... y del annus horribilis en que esta temporada se está tornando para Pep Guardiola. Como era de esperar, también las redes han hecho su 'trabajo' con el desempeño del técnico catalán. Incapaz de encontrar una solución y sin meter a Haaland en nin

In [28]:
data_df.iloc[-3].values

array(['La Razón',
       'https://www.larazon.es/deportes/futbol/claves-nuevo-real-madrid-bronca-442-canterano-jugador-que-vale-todo_2025022067b6fd23b1a8db0001ce6b47.html',
       'Las claves del nuevo Real Madrid: una bronca, el 4-4-2, Mbappé, un canterano y un jugador que vale para todo ',
       datetime.date(2025, 2, 20),
       'Deportes  Fútbol España Local Opinión Internacional Economía Cultura Sociedad Deportes Gente A Tu Salud Televisión Ciencia Motor Lifestyle Emergente Premios LR Juegos y pasatiempos 25 Aniversario De Compras La tienda de La Razón Lea el periódico en PDF Stories La Razón Suplementos en PDF Promociones Directo Última hora del estado de salud del Papa Directo Última hora de la guerra de Ucrania: Musk arremete contra Zelenski y Trump le pide que firme ya el acuerdo de las tierras raras Fútbol Carlo Ancelotti se ha pasado lo que llevamos de temporada buscando el punto de inflexión del equipo. Porque el Real Madrid ha ido mezclando momentos buenos con otros regu

## Filtrar textos

### Buscar textos que tengan al menos una palabra clave

In [29]:
# Filtrar por palabras clave
keywords = ['universidad', 'oviedo']

# Regex pattern
pattern = '|'.join(keywords)

# Filtrar dataframe
filtered_df = data_df[data_df['texto'].str.contains(pattern, case=False, na=False)].reset_index(drop=True)

filtered_df

,fuente,url,titulo,fecha_publicacion,texto
0,La Razón,https://www.larazon.es/asturias/fernando-laser...,"Fernando Laserna Cocina, nombrado nuevo Fiscal...",2025-02-21,Asturias España Local Opinión Internacional Ec...
1,La Razón,https://www.larazon.es/andalucia/sumar-designa...,Sumar designa este sábado a Esperanza Gómez y ...,2025-02-21,Andalucía España Local Opinión Internacional E...
2,La Razón,https://www.larazon.es/castilla-y-leon/burgos-...,Burgos acoge el Foro de Trabajo de Saborea Esp...,2025-02-21,Castilla y León España Local Opinión Internaci...
3,La Razón,https://www.larazon.es/castilla-la-mancha/trab...,<![CDATA[Un trabajador resulta herido grave tr...,2025-02-21,Hoy Suscríbase a nuestro canal de WhatsApp Ser...
4,La Razón,https://www.larazon.es/castilla-la-mancha/sema...,"Fin de semana pasado por lluvia, tormentas y v...",2025-02-21,Hoy Suscríbase a nuestro canal de WhatsApp Ser...
...,...,...,...,...,...
804,La Razón,https://www.larazon.es/comunidad-valenciana/po...,"Quién es Arturo Torró, exalcalde de Gandía y f...",2025-02-20,Comunidad Valenciana España Local Opinión Inte...
805,La Razón,https://www.larazon.es/cultura/irse-cerros-ube...,<![CDATA['Irse por los cerros de Úbeda': Dónde...,2025-02-20,Hoy Suscríbase a nuestro canal de WhatsApp Car...
806,ElDiario.es,https://www.eldiario.es/catalunya/muere-viqui-...,NaN,2025-02-20,"Hoy hablamos de... Por favor, actualiza tus da..."
807,La Razón,https://www.larazon.es/economia/eae-business-s...,"EAE Business School Madrid, centro de Innovaci...",2025-02-20,Economía España Local Opinión Internacional Ec...


In [30]:
filtered_df['url'].values[0:100]

array(['https://www.larazon.es/asturias/fernando-laserna-cocina-nombrado-nuevo-fiscal-delegado-comunidad-autonoma-personas-discapacidad-mayores_2025022167b7b4e5500f9600011436eb.html',
       'https://www.larazon.es/andalucia/sumar-designa-este-sabado-esperanza-gomez-raul-garcia-como-coordinadores-andaluces_2025022167b87720417ec20001015cd6.html',
       'https://www.larazon.es/castilla-y-leon/burgos-acoge-foro-trabajo-saborea-espana-participacion-14-destinos-gastronomicos_2025022167b85939417ec20001012020.html',
       'https://www.larazon.es/castilla-la-mancha/trabajador-resulta-herido-grave-caerle-encima-tolva-fabrica-villacanas-toledo/20250221/1168744.html',
       'https://www.larazon.es/castilla-la-mancha/semana-pasado-lluvia-tormentas-vuelta-frio-castilla-mancha-aemet/20250221/1168741.html',
       'https://www.larazon.es/castilla-la-mancha/acabar-espana-vaciada-empueblate-ofrece-soluciones-exito-impulsar-repoblar-pueblos/20250221/1168766.html',
       'https://www.larazon.es/cultu

## Buscar textos que tengan todas las palabras clave

In [ ]:
# Lista de palabras clave
keywords = ['universidad', 'oviedo']

# Crear una condición de filtro para cada palabra clave
filter_condition = data_df['texto'].str.contains(keywords[0], case=False, na=False)

for keyword in keywords[1:]:
    filter_condition &= data_df['texto'].str.contains(keyword, case=False, na=False)

# Aplicar filtro
filtered_df_all = data_df[filter_condition].reset_index(drop=True)

filtered_df_all

,fuente,url,titulo,fecha_publicacion,texto
0,La Razón,https://www.larazon.es/galicia/quien-era-alfre...,"<![CDATA[¿Quién era Alfredo Brañas, el ejemplo...",2025-02-21,Galicia España Local Opinión Internacional Eco...
1,La Razón,https://www.larazon.es/asturias/fernando-laser...,"Fernando Laserna Cocina, nombrado nuevo Fiscal...",2025-02-21,Asturias España Local Opinión Internacional Ec...
2,Nortes,https://www.nortes.me/2025/02/21/una-estatua-p...,NaN,2025-02-21,Las políticas asturianas de memoria han sido a...
3,Radiotelevisión del Principado de Asturias (RTPA),https://www.rtpa.es/noticias-asturias/2025-02-...,La sierense Susana Madera será la nueva viceco...,2025-02-21,Ver la transcripción del vídeo de esta noticia...
4,Nortes,https://www.nortes.me/articulos-recientes/,NaN,2025-02-21,El general Augusto Pinochet quería ser un augu...
...,...,...,...,...,...
152,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/oviedo/...,NaN,NaT,Andrew Silin y Kate Yakusheva se conocían de c...
153,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/siero/2...,NaN,NaT,"El alcalde de Siero, Ángel García, se ha mostr..."
154,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/turismo...,NaN,NaT,La Comarca Vaqueira toma su nombre de los vaqu...
155,La Voz de Asturias,https://www.lavozdeasturias.es/noticia/turismo...,NaN,NaT,El concejo de Oviedo tiene en su corazón la ci...


In [ ]:
def mostrar_informacion(data_df):

    for index, fila in data_df.iterrows():
        if pd.isna(fila['fecha_publicacion']):
            fecha = 'Desconocido'
        else:
            fecha = fila['fecha_publicacion'].strftime('%Y/%m/%d')
        fuente = fila['fuente']

        titulo = 'Desconocido'
        if pd.notna(fila['titulo']):
            titulo = fila['titulo']

        print(titulo)
        print(fila['url'])
        print(f'{fecha} - {fuente}')
        print()

mostrar_informacion(filtered_df_all)

<![CDATA[¿Quién era Alfredo Brañas, el ejemplo que el PP pone como modelo de unidad y respeto?]]>
https://www.larazon.es/galicia/quien-era-alfredo-branas-ejemplo-que-pone-como-modelo-unidad-respeto-p7m_2025022167b868cc417ec20001013b82.html
2025/02/21 - La Razón

Fernando Laserna Cocina, nombrado nuevo Fiscal Delegado en la comunidad autónoma de Personas con Discapacidad y Mayores
https://www.larazon.es/asturias/fernando-laserna-cocina-nombrado-nuevo-fiscal-delegado-comunidad-autonoma-personas-discapacidad-mayores_2025022167b7b4e5500f9600011436eb.html
2025/02/21 - La Razón

Desconocido
https://www.nortes.me/2025/02/21/una-estatua-para-el-rector-alas/
2025/02/21 - Nortes

La sierense Susana Madera será la nueva viceconsejera de Medio Ambiente
https://www.rtpa.es/noticias-asturias/2025-02-21/La-sierense-Susana-Madera-sera-la-nueva-viceconsejera-de-Medio-Ambiente_111740125730.html
2025/02/21 - Radiotelevisión del Principado de Asturias (RTPA)

Desconocido
https://www.nortes.me/articulos-re